# Tune `linear_lr`

Linear (RF top-k + T² + hub×hub interactions) + elastic-net logistic (saga). Repeated stratified CV on the train split;
writes [`data/processed/tuned/linear_lr.json`](../data/processed/tuned/linear_lr.json).

Tune all models: [`tune_all.ipynb`](tune_all.ipynb).


**Classifier:** `elastic_net_lr` (saga + `penalty="elasticnet"`) in `scripts/secom_pipelines.py`.

**Stage 1 (hyperparameters):** RF top-k (`top_k`), hub count (`n_hubs`), classifier `C`, `l1_ratio`. Select by **max mean PR AUC**.

**Stage 2 (threshold):** sweep `THRESHOLD_GRID` on the same CV folds; select threshold that **minimizes mean BER**.

Shared sensor branch adds an **isolation forest** `decision_function` score after T² and hub interactions (unsupervised, refit per CV fold). Grid includes `isolation_forest__n_estimators`.

After hubs: **neighbor_fail_rate** (RF-weighted kNN mean train-label rate; tune `neighbor_fail_rate__n_neighbors`), then isolation forest.


In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd

_cwd = Path.cwd()
REPO_ROOT = _cwd.parent if _cwd.name == "tuning" else _cwd
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import scripts.tuning.registry as tuning_registry

importlib.reload(tuning_registry)
from scripts.secom_pipelines import TARGET_COL, feature_columns, load_mart, split_train_test
from scripts.tuning.registry import (
    MODEL_SPECS,
    fit_with_progress,
    run_grid_search,
    save_tuned_params,
    summarize_cv_search,
    tune_classifier_threshold,
    tuned_params_path,
)

MODEL_ID = "linear_lr"
spec = MODEL_SPECS[MODEL_ID]


In [2]:
df = load_mart()
feature_cols = feature_columns(df)
train_df, test_df = split_train_test(df)
X_train = train_df[feature_cols]
y_train = train_df[TARGET_COL].astype(int)
print(len(X_train), "train rows", len(test_df), "test rows (holdout, not used here)")


1253 train rows 314 test rows (holdout, not used here)


In [3]:
param_grid = spec.make_param_grid()
pd.DataFrame([{k: v} for k, v in param_grid.items()])


,preprocess__sensor_branch__select_t2_hubs__top_k,preprocess__sensor_branch__select_t2_hubs__n_hubs,classifier__C,classifier__l1_ratio
0,"[20, 40, 60, 80]",NaN,NaN,NaN
1,NaN,"[10, 15, 20]",NaN,NaN
2,NaN,NaN,"[0.001, 0.005, 0.01, 0.05]",NaN
3,NaN,NaN,NaN,[1.0]


In [4]:
search, n_candidates, n_splits, total_fits = run_grid_search(spec, X_train, y_train)
print(f"{MODEL_ID}: {n_candidates} candidates x {n_splits} folds = {total_fits} fits")
search = fit_with_progress(search, X_train, y_train)


linear_lr: 48 candidates x 5 folds = 240 fits


GridSearchCV 240 fits:   0%|          | 0/240 [00:00<?, ?it/s]

  0%|          | 0/240 [00:00<?, ?it/s]

Fitting 5 folds for each of 48 candidates, totalling 240 fits


In [5]:
cv_summary, fold_results, aggregated = summarize_cv_search(search, spec)
print("Stage 1 best (mean PR AUC):")
display(aggregated.head(10))


Stage 1 best (mean PR AUC):


,top_k,n_hubs,c,l1_ratio,mean_ber_percent,std_ber_percent,mean_balanced_accuracy,mean_true_positive_percent,std_true_positive_percent,mean_true_negative_percent,std_true_negative_percent,mean_roc_auc,std_roc_auc,mean_pr_auc,std_pr_auc
18,40,15,0.01,1.0,36.199095,5.668207,0.638009,65.294118,13.240604,62.307692,2.626033,0.688176,0.041848,0.180304,0.038108
14,40,10,0.01,1.0,37.416415,4.216688,0.625836,63.970588,9.530501,61.196581,3.548810,0.683783,0.038058,0.177351,0.033332
6,20,15,0.01,1.0,35.727124,2.797774,0.642729,66.323529,4.214974,62.222222,2.334444,0.689266,0.029808,0.171907,0.018144
22,40,20,0.01,1.0,36.070890,4.903501,0.639291,65.294118,11.566810,62.564103,2.426508,0.690249,0.040342,0.169711,0.049095
42,80,15,0.01,1.0,36.626445,6.053294,0.633736,65.294118,11.566810,61.452991,4.897965,0.681957,0.048753,0.169592,0.048988
2,20,10,0.01,1.0,36.806184,2.621206,0.631938,63.823529,5.534767,62.564103,2.270997,0.684832,0.029767,0.168581,0.015480
30,60,15,0.01,1.0,36.044180,6.273253,0.639558,66.544118,13.791421,61.367521,4.329547,0.683029,0.047046,0.167881,0.048671
46,80,20,0.01,1.0,36.567685,5.516058,0.634323,64.044118,11.463044,62.820513,2.661951,0.678761,0.049311,0.167407,0.053655
34,60,20,0.01,1.0,35.942685,5.169197,0.640573,65.294118,11.566810,62.820513,2.402302,0.679588,0.048440,0.166098,0.050941
10,20,20,0.01,1.0,36.138449,3.266627,0.638616,65.073529,4.236724,62.649573,2.842443,0.690388,0.035104,0.162822,0.024843


In [6]:
threshold_result = tune_classifier_threshold(spec, X_train, y_train, cv_summary)
print(f"Stage 2 best threshold: {threshold_result['best_threshold']:.2f}")
print(f"  mean BER at threshold: {threshold_result['mean_ber_percent']:.2f}%")
display(threshold_result["per_threshold_mean_ber"].head(10))


Threshold CV folds:   0%|          | 0/5 [00:00<?, ?it/s]

Stage 2 best threshold: 0.50
  mean BER at threshold: 36.20%


,threshold,mean_ber_percent
0,0.50,36.199095
1,0.55,37.601181
2,0.45,40.637255
3,0.60,42.304236
4,0.40,42.941805
5,0.35,44.029663
6,0.65,44.285131
7,0.70,46.471845
8,0.25,48.076923
9,0.30,48.142283


In [7]:
payload = save_tuned_params(
    spec,
    cv_summary,
    fold_results,
    aggregated,
    threshold_result=threshold_result,
)
out_path = tuned_params_path(MODEL_ID)
print(f"Wrote {out_path}")
payload["grid_search_best_params"]


Wrote /home/troy/SECOM/data/processed/tuned/linear_lr.json


{'preprocess__sensor_branch__select_t2_hubs__top_k': 40,
 'preprocess__sensor_branch__select_t2_hubs__n_hubs': 15,
 'classifier__C': 0.01,
 'classifier__l1_ratio': 1.0}